# 中证800 V73 监控触发组合动作实验

目标：沿着 V72 的排序健康诊断继续，验证“监控触发后暂停/降仓”是否能改善 top8 alpha 路径。

核心原则：

- 不重新训练模型。
- 不用当月健康状态决定当月仓位，避免未来函数。
- 每个月的仓位动作只使用上一个可观察月份的 V72 监控结果。
- 先验证规则是否改善 alpha 路径，再考虑写入 JoinQuant 回测文件。

输入：V72 输出文件，至少需要 `v72_ranking_health_state.csv`。如果有 `v72_ranking_metrics_monthly.csv` 和 `v72_model_meta.csv` 会输出更完整报告。

输出：

- `v73_monitor_policy_summary.csv`
- `v73_monitor_policy_monthly.csv`
- `v73_monitor_trigger_diagnostics.csv`
- `v73_recommended_policy.csv`


## 0. 导入与进度条

In [ ]:
import os
import warnings
import builtins as _bi
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = _bi.max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


## 1. 配置与数据路径

默认在当前目录、V72 输出目录、Downloads 里寻找 V72 CSV。聚宽研究环境里建议把 V72 notebook 和 V73 notebook 放在同一个工作目录，先跑完 V72 再跑 V73。

In [ ]:
PROJECT_DIR = Path.cwd()
OUT_DIR = PROJECT_DIR / "csi800_ml_v73_monitor_policy_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

V72_OUT_DIR_CANDIDATES = [
    PROJECT_DIR / "csi800_ml_v72_ranking_quality_outputs",
    PROJECT_DIR,
    Path("/Users/youzou/Downloads"),
]

HEALTH_FILE = "v72_ranking_health_state.csv"
MONTHLY_FILE = "v72_ranking_metrics_monthly.csv"
META_FILE = "v72_model_meta.csv"

TOP_ALPHA_COL = "pred_top8_alpha"
STATE_COL = "ranking_health_state"
PERCENTILE_COL = "pred_top8_alpha_random_percentile"
DATE_COL = "rebalance_date"
MODEL_COL = "model_tag"

# 年度滚动生产路径：每年使用上一年年底训练完成的模型。
ANNUAL_MODEL_BY_YEAR = {
    2022: "exp_2021_12",
    2023: "exp_2022_12",
    2024: "exp_2023_12",
    2025: "exp_2024_12",
    2026: "exp_2025_12",
}

# 如果 V72 没有 exp_2025_12，可以回退用 exp_2024_12 跑 2026 观察。
ANNUAL_FALLBACK_MODEL_BY_YEAR = {
    2026: "exp_2024_12",
}


## 2. 读取 V72 输出

In [ ]:
def first_existing_file(filename):
    checked = []
    for base in V72_OUT_DIR_CANDIDATES:
        p = Path(base) / filename
        checked.append(str(p))
        if p.exists():
            return p
    raise IOError("file not found: %s\nchecked:\n%s" % (filename, "\n".join(checked)))


def safe_read_csv(filename, required=True):
    try:
        p = first_existing_file(filename)
    except IOError:
        if required:
            raise
        print("optional file missing:", filename)
        return pd.DataFrame(), None
    df = pd.read_csv(p)
    print("loaded", filename, df.shape, "from", p)
    return df, p


health_df, health_path = safe_read_csv(HEALTH_FILE, required=True)
monthly_df, monthly_path = safe_read_csv(MONTHLY_FILE, required=False)
meta_df, meta_path = safe_read_csv(META_FILE, required=False)

for df in [health_df, monthly_df, meta_df]:
    if isinstance(df, pd.DataFrame) and DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL])

need_cols = [DATE_COL, MODEL_COL, TOP_ALPHA_COL, STATE_COL, PERCENTILE_COL]
missing = [c for c in need_cols if c not in health_df.columns]
if missing:
    raise ValueError("health file missing columns: " + ",".join(missing))

health_df = health_df.copy()
health_df[DATE_COL] = pd.to_datetime(health_df[DATE_COL])
health_df["year"] = health_df[DATE_COL].dt.year
health_df["month"] = health_df[DATE_COL].dt.strftime("%Y-%m")
health_df = health_df.sort_values([MODEL_COL, DATE_COL]).reset_index(drop=True)

print("date range:", health_df[DATE_COL].min(), health_df[DATE_COL].max())
print("models:", sorted(health_df[MODEL_COL].unique()))
display_df(health_df[[DATE_COL, MODEL_COL, "phase", TOP_ALPHA_COL, PERCENTILE_COL, STATE_COL]].sort_values([DATE_COL, MODEL_COL]), 20)


## 3. 构造评估路径

两类路径：

1. `per_model`：每个模型各自完整 OOS 测试区间，评估规则在单模型上的效果。
2. `annual_retrain`：模拟年度滚动生产，2022 用 exp_2021_12，2023 用 exp_2022_12，以此类推。

In [ ]:
def build_per_model_paths(health):
    out = health.copy()
    out["path_name"] = "per_model__" + out[MODEL_COL].astype(str)
    out["path_type"] = "per_model"
    return out


def build_annual_retrain_path(health):
    parts = []
    for year in sorted(health["year"].unique()):
        model = ANNUAL_MODEL_BY_YEAR.get(int(year), None)
        sub = pd.DataFrame()
        if model is not None:
            sub = health[(health["year"] == year) & (health[MODEL_COL] == model)].copy()
        if sub.empty and int(year) in ANNUAL_FALLBACK_MODEL_BY_YEAR:
            model = ANNUAL_FALLBACK_MODEL_BY_YEAR[int(year)]
            sub = health[(health["year"] == year) & (health[MODEL_COL] == model)].copy()
        if not sub.empty:
            parts.append(sub)
    if not parts:
        return pd.DataFrame()
    out = pd.concat(parts, axis=0, ignore_index=True, sort=False)
    out["path_name"] = "annual_retrain"
    out["path_type"] = "annual_retrain"
    return out.sort_values(DATE_COL).reset_index(drop=True)


path_df = pd.concat([build_per_model_paths(health_df), build_annual_retrain_path(health_df)], axis=0, ignore_index=True, sort=False)
path_df = path_df.sort_values(["path_name", DATE_COL]).reset_index(drop=True)
print("paths:", path_df["path_name"].nunique(), "rows:", len(path_df))
print(path_df.groupby(["path_type", "path_name"])[DATE_COL].agg(["min", "max", "count"]).to_string())


## 4. 监控动作规则

所有规则都使用上月状态，因此本月仓位是 `prev_*` 决定的。

- `raw_full`: 不做监控动作，始终满仓。
- `prev_red_cash`: 上月 red，本月空仓。
- `prev_red_half`: 上月 red，本月半仓。
- `prev_red_yellow_cash`: 上月 red/yellow，本月空仓。
- `prev_pct25_cash`: 上月随机分位 <25%，本月空仓。
- `prev_pct25_half`: 上月随机分位 <25%，本月半仓。
- `prev_red_or_pct25_cash`: 上月 red 或随机分位 <25%，本月空仓。
- `prev_two_red_cash`: 连续两个月 red 后，本月空仓。
- `prev_two_weak_cash`: 过去两个月至少两次 red/yellow 或分位 <25%，本月空仓。

In [ ]:
def add_previous_monitor_features(df):
    out = df.sort_values(DATE_COL).copy()
    out["prev_state"] = out[STATE_COL].shift(1)
    out["prev_pct"] = out[PERCENTILE_COL].shift(1)
    out["prev_alpha"] = out[TOP_ALPHA_COL].shift(1)
    out["prev_is_red"] = out["prev_state"].eq("red")
    out["prev_is_yellow_or_red"] = out["prev_state"].isin(["yellow", "red"])
    out["prev_pct_lt25"] = out["prev_pct"] < 0.25
    out["prev_weak_flag"] = out["prev_is_yellow_or_red"] | out["prev_pct_lt25"]
    out["prev2_state"] = out[STATE_COL].shift(2)
    out["prev2_pct"] = out[PERCENTILE_COL].shift(2)
    out["prev_two_red"] = out["prev_is_red"] & out["prev2_state"].eq("red")
    out["prev2_weak_flag"] = out["prev2_state"].isin(["yellow", "red"]) | (out["prev2_pct"] < 0.25)
    out["prev_two_weak"] = out["prev_weak_flag"].fillna(False) & out["prev2_weak_flag"].fillna(False)
    return out


def exposure_for_policy(df, policy):
    if policy == "raw_full":
        return pd.Series(1.0, index=df.index)
    if policy == "prev_red_cash":
        return np.where(df["prev_is_red"].fillna(False), 0.0, 1.0)
    if policy == "prev_red_half":
        return np.where(df["prev_is_red"].fillna(False), 0.5, 1.0)
    if policy == "prev_red_yellow_cash":
        return np.where(df["prev_is_yellow_or_red"].fillna(False), 0.0, 1.0)
    if policy == "prev_pct25_cash":
        return np.where(df["prev_pct_lt25"].fillna(False), 0.0, 1.0)
    if policy == "prev_pct25_half":
        return np.where(df["prev_pct_lt25"].fillna(False), 0.5, 1.0)
    if policy == "prev_red_or_pct25_cash":
        flag = df["prev_is_red"].fillna(False) | df["prev_pct_lt25"].fillna(False)
        return np.where(flag, 0.0, 1.0)
    if policy == "prev_two_red_cash":
        return np.where(df["prev_two_red"].fillna(False), 0.0, 1.0)
    if policy == "prev_two_weak_cash":
        return np.where(df["prev_two_weak"].fillna(False), 0.0, 1.0)
    raise ValueError("unknown policy: " + str(policy))


POLICIES = [
    "raw_full",
    "prev_red_cash",
    "prev_red_half",
    "prev_red_yellow_cash",
    "prev_pct25_cash",
    "prev_pct25_half",
    "prev_red_or_pct25_cash",
    "prev_two_red_cash",
    "prev_two_weak_cash",
]


## 5. 策略路径评估函数

In [ ]:
def calc_max_drawdown(equity):
    s = pd.Series(equity).astype(float)
    if s.empty:
        return np.nan
    peak = s.cummax()
    dd = s / peak - 1.0
    return float(dd.min())


def summarize_return_series(ret):
    r = pd.Series(ret).astype(float).fillna(0.0)
    if r.empty:
        return {
            "months": 0,
            "cum_ret": np.nan,
            "mean_ret": np.nan,
            "win_rate": np.nan,
            "max_drawdown": np.nan,
            "worst_month": np.nan,
            "best_month": np.nan,
        }
    equity = (1.0 + r).cumprod()
    return {
        "months": int(len(r)),
        "cum_ret": float(equity.iloc[-1] - 1.0),
        "mean_ret": float(r.mean()),
        "win_rate": float((r > 0).mean()),
        "max_drawdown": calc_max_drawdown(equity),
        "worst_month": float(r.min()),
        "best_month": float(r.max()),
    }


def evaluate_one_path(path_name, df):
    base = add_previous_monitor_features(df)
    rows = []
    monthly_parts = []
    for policy in POLICIES:
        tmp = base.copy()
        tmp["policy"] = policy
        tmp["exposure"] = exposure_for_policy(tmp, policy)
        tmp["policy_alpha"] = tmp[TOP_ALPHA_COL].fillna(0.0) * tmp["exposure"]
        tmp["raw_alpha"] = tmp[TOP_ALPHA_COL].fillna(0.0)
        tmp["inactive"] = tmp["exposure"] < 0.999
        tmp["missed_positive"] = tmp["inactive"] & (tmp["raw_alpha"] > 0)
        tmp["avoided_negative"] = tmp["inactive"] & (tmp["raw_alpha"] < 0)
        tmp["path_name"] = path_name
        monthly_parts.append(tmp)

        s = summarize_return_series(tmp["policy_alpha"])
        raw = summarize_return_series(tmp["raw_alpha"])
        s.update({
            "path_name": path_name,
            "path_type": tmp["path_type"].iloc[0] if "path_type" in tmp.columns and len(tmp) else "",
            "policy": policy,
            "raw_cum_ret": raw["cum_ret"],
            "raw_max_drawdown": raw["max_drawdown"],
            "excess_cum_vs_raw": s["cum_ret"] - raw["cum_ret"] if pd.notnull(s["cum_ret"]) and pd.notnull(raw["cum_ret"]) else np.nan,
            "dd_improvement_vs_raw": s["max_drawdown"] - raw["max_drawdown"] if pd.notnull(s["max_drawdown"]) and pd.notnull(raw["max_drawdown"]) else np.nan,
            "active_rate": float((tmp["exposure"] > 0).mean()) if len(tmp) else np.nan,
            "avg_exposure": float(tmp["exposure"].mean()) if len(tmp) else np.nan,
            "inactive_months": int(tmp["inactive"].sum()),
            "avoided_negative_months": int(tmp["avoided_negative"].sum()),
            "missed_positive_months": int(tmp["missed_positive"].sum()),
            "avoided_negative_alpha_sum": float((-tmp.loc[tmp["avoided_negative"], "raw_alpha"]).sum()) if len(tmp) else 0.0,
            "missed_positive_alpha_sum": float(tmp.loc[tmp["missed_positive"], "raw_alpha"].sum()) if len(tmp) else 0.0,
        })
        rows.append(s)
    return pd.DataFrame(rows), pd.concat(monthly_parts, axis=0, ignore_index=True, sort=False)


## 6. 运行 V73 监控策略实验

In [ ]:
summary_parts = []
monthly_parts = []
path_names = sorted(path_df["path_name"].dropna().unique())
for path_name in progress_iter(path_names, total=len(path_names), desc="monitor policies"):
    one = path_df[path_df["path_name"] == path_name].copy()
    if one.empty:
        continue
    summary_one, monthly_one = evaluate_one_path(path_name, one)
    summary_parts.append(summary_one)
    monthly_parts.append(monthly_one)

policy_summary_df = pd.concat(summary_parts, axis=0, ignore_index=True, sort=False)
policy_monthly_df = pd.concat(monthly_parts, axis=0, ignore_index=True, sort=False)

# 排序：先看年度滚动路径，再看单模型；同一路径内优先看回撤改善和累计收益。
policy_summary_df = policy_summary_df.sort_values(
    ["path_type", "path_name", "dd_improvement_vs_raw", "cum_ret"],
    ascending=[True, True, False, False]
).reset_index(drop=True)

summary_path = OUT_DIR / "v73_monitor_policy_summary.csv"
monthly_path = OUT_DIR / "v73_monitor_policy_monthly.csv"
policy_summary_df.to_csv(summary_path, index=False)
policy_monthly_df.to_csv(monthly_path, index=False)
print("saved:", summary_path)
print("saved:", monthly_path)

display_df(policy_summary_df, 60)


## 7. 触发质量诊断

这里看监控触发到底是在避免亏损，还是误杀收益。

In [ ]:
trigger_rows = []
for path_name, g in policy_monthly_df.groupby("path_name"):
    raw = g[g["policy"] == "raw_full"].copy()
    raw_ret = raw[[DATE_COL, "raw_alpha"]].rename(columns={"raw_alpha": "raw_alpha_ref"})
    for policy in [p for p in POLICIES if p != "raw_full"]:
        one = g[g["policy"] == policy].merge(raw_ret, on=DATE_COL, how="left")
        inactive = one[one["inactive"]].copy()
        if inactive.empty:
            trigger_rows.append({
                "path_name": path_name,
                "path_type": one["path_type"].iloc[0] if len(one) else "",
                "policy": policy,
                "trigger_months": 0,
                "trigger_rate": 0.0,
                "avoided_negative_rate": np.nan,
                "missed_positive_rate": np.nan,
                "trigger_raw_alpha_mean": np.nan,
                "trigger_raw_alpha_sum": 0.0,
            })
            continue
        trigger_rows.append({
            "path_name": path_name,
            "path_type": one["path_type"].iloc[0] if len(one) else "",
            "policy": policy,
            "trigger_months": int(len(inactive)),
            "trigger_rate": float(len(inactive) / float(len(one))) if len(one) else np.nan,
            "avoided_negative_rate": float((inactive["raw_alpha_ref"] < 0).mean()),
            "missed_positive_rate": float((inactive["raw_alpha_ref"] > 0).mean()),
            "trigger_raw_alpha_mean": float(inactive["raw_alpha_ref"].mean()),
            "trigger_raw_alpha_sum": float(inactive["raw_alpha_ref"].sum()),
        })

trigger_diag_df = pd.DataFrame(trigger_rows).sort_values(["path_type", "path_name", "trigger_raw_alpha_mean"])
trigger_path = OUT_DIR / "v73_monitor_trigger_diagnostics.csv"
trigger_diag_df.to_csv(trigger_path, index=False)
print("saved:", trigger_path)
display_df(trigger_diag_df, 80)


## 8. 推荐规则选择

一个规则如果只是降低回撤但大幅牺牲累计收益，不适合生产。这里用保守规则筛选：

- 年度滚动路径优先。
- 不能显著损害累计收益。
- 回撤改善优先。
- 触发月份不能太多，避免策略长期空仓。

In [ ]:
def choose_recommended(summary):
    rows = []
    for path_name, g in summary.groupby("path_name"):
        raw = g[g["policy"] == "raw_full"]
        if raw.empty:
            continue
        raw_cum = float(raw["cum_ret"].iloc[0])
        raw_dd = float(raw["max_drawdown"].iloc[0])
        cand = g[g["policy"] != "raw_full"].copy()
        if cand.empty:
            continue
        cand["cum_ret_loss_vs_raw"] = raw_cum - cand["cum_ret"]
        cand["mdd_abs_improvement"] = cand["max_drawdown"] - raw_dd
        # 不接受累计收益损失超过 10 个百分点，除非回撤改善超过 10 个百分点。
        ok = cand[(cand["cum_ret_loss_vs_raw"] <= 0.10) | (cand["mdd_abs_improvement"] >= 0.10)].copy()
        if ok.empty:
            ok = cand.copy()
        ok["selection_score"] = ok["mdd_abs_improvement"] * 2.0 + ok["excess_cum_vs_raw"] - ok["inactive_months"] * 0.002
        best = ok.sort_values("selection_score", ascending=False).head(1).copy()
        rows.append(best)
    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, axis=0, ignore_index=True, sort=False)

recommended_df = choose_recommended(policy_summary_df)
rec_path = OUT_DIR / "v73_recommended_policy.csv"
recommended_df.to_csv(rec_path, index=False)
print("saved:", rec_path)
display_df(recommended_df[["path_name", "path_type", "policy", "months", "cum_ret", "raw_cum_ret", "max_drawdown", "raw_max_drawdown", "excess_cum_vs_raw", "dd_improvement_vs_raw", "inactive_months", "selection_score"]], 50)


## 9. 年度滚动路径重点展示

In [ ]:
annual = policy_summary_df[policy_summary_df["path_name"] == "annual_retrain"].copy()
cols = ["policy", "months", "cum_ret", "mean_ret", "win_rate", "max_drawdown", "excess_cum_vs_raw", "dd_improvement_vs_raw", "avg_exposure", "inactive_months", "avoided_negative_months", "missed_positive_months", "avoided_negative_alpha_sum", "missed_positive_alpha_sum"]
if not annual.empty:
    display_df(annual[cols].sort_values(["dd_improvement_vs_raw", "cum_ret"], ascending=[False, False]), 40)
else:
    print("annual_retrain path is empty; check ANNUAL_MODEL_BY_YEAR or V72 model coverage.")

annual_monthly = policy_monthly_df[policy_monthly_df["path_name"] == "annual_retrain"].copy()
if not annual_monthly.empty:
    show_cols = [DATE_COL, MODEL_COL, STATE_COL, PERCENTILE_COL, TOP_ALPHA_COL, "policy", "exposure", "policy_alpha", "prev_state", "prev_pct"]
    show_policy = annual_monthly["policy"].isin(["raw_full", "prev_pct25_cash", "prev_red_cash", "prev_red_or_pct25_cash"])
    show_df = annual_monthly[show_policy].sort_values([DATE_COL, "policy"])
    display_df(show_df[show_cols], 120)


## 10. 结论模板

运行后重点看：

1. `annual_retrain` 上哪个规则能改善回撤，同时不明显牺牲累计收益。
2. `prev_pct25_cash` 是否仍是比 `red` 更稳的触发规则。
3. 触发月份里到底是避免负收益多，还是误杀正收益多。
4. 如果所有规则都牺牲收益且回撤改善有限，则 V72 监控只适合作为人工复盘信号，不应自动交易。

生产化前的最低要求：

- V73 规则在离线路径有效。
- 用相同规则写入 JoinQuant 回测文件，确认真实回测路径同向改善。
- 日志必须记录上月 state、percentile、本月 exposure、触发原因。